In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, ListedColormap , LogNorm
from matplotlib.cm import ScalarMappable
import numpy as np
import os
from tqdm import tqdm

# Set Matplotlib to display images inline in the Notebook
%matplotlib inline

In [ ]:
# --- Configuration ---
# Set your file paths and analysis parameters here.

# Directory containing all prediction .csv files
PREDICTIONS_DIR = 'predictions/'

# File path for the locations data (containing location_id and geometry)
LOCATIONS_FILE = 'locs.csv'

# File path for the Switzerland GeoJSON boundary file
SWITZERLAND_GEOJSON = 'switzerland.geojson'

# File name prefix for the final saved composite image
OUTPUT_FILE_PREFIX = 'comparison_accuracy'

# =======================================================
# --- Core Analysis Mode ---
# Set to True -> Analyze all users in the test set
# Set to False -> Analyze only the TARGET_USER_ID below
ANALYZE_ALL_USERS = True

# (Only effective when ANALYZE_ALL_USERS is False)
TARGET_USER_ID = 700 

# --- Set the K for Top-K accuracy analysis ---
TOP_K = 5
# =======================================================

In [ ]:
def prepare_locations_data(locations_file):
    """Loads and prepares the location GeoDataFrame."""
    try:
        locs_df = pd.read_csv(locations_file)
        locs_df['geometry'] = locs_df['geometry'].apply(wkt.loads)
        locs_gdf = gpd.GeoDataFrame(locs_df, geometry='geometry', crs="EPSG:4326")
        return locs_gdf.to_crs("EPSG:3857")
    except FileNotFoundError:
        print(f"Error: Location file not found: {locations_file}")
        return None

def get_switzerland_boundaries(geojson_path):
    """Loads the Switzerland boundary GeoJSON."""
    try:
        ch_boundaries = gpd.read_file(geojson_path)
        return ch_boundaries.to_crs("EPSG:3857")
    except Exception as e:
        print(f"Warning: Could not load Switzerland boundaries: {e}")
        return None

def load_and_process_data(pred_dir, locs_gdf, analyze_all=False, target_user_id=None, top_k=5):
    """
    Loads data and calculates two metrics:
    1. Visit Frequency (%) for the 'True' data.
    2. Top-K Accuracy (%) for the 'Prediction' data.
    """
    if locs_gdf is None: raise ValueError("Location geodata failed to load; cannot continue.")
    prefixes, models = ['dtepr', 'epr'], ['LSTM', 'Mamba', 'MHSA']
    processed_gdfs = {}
    
    mode_str = "All Users" if analyze_all else f"User ID: {target_user_id}"
    print(f"Mode: {mode_str} | Processing Top-{top_k} data for Frequency vs. Accuracy...")

    for prefix in prefixes:
        try:
            sample_file = os.path.join(pred_dir, f"{prefix}_{models[0]}_test_predictions.csv")
            full_df = pd.read_csv(sample_file)
            data_subset = full_df if analyze_all else full_df[full_df['user_id'] == target_user_id]
            if not data_subset.empty:
                true_counts = data_subset['true_label'].value_counts().reset_index(); true_counts.columns = ['location_id', 'count']
                true_counts['count'] = (true_counts['count'] / len(data_subset)) * 100
                processed_gdfs[f'{prefix.upper()} True'] = locs_gdf.merge(true_counts, on='location_id', how='inner')
        except FileNotFoundError: print(f"Warning: File not found: {sample_file}, skipping {prefix.upper()} True data."); continue
        
        for model in models:
            filepath = os.path.join(pred_dir, f"{prefix}_{model}_test_predictions.csv")
            try:
                full_pred_df = pd.read_csv(filepath)
                pred_cols = [f'pred_{i}' for i in range(1, top_k + 1)]
                if not all(col in full_pred_df.columns for col in pred_cols):
                    print(f"Warning: File {filepath} is missing Top-{top_k} prediction columns, skipping."); continue
                
                data_subset = full_pred_df if analyze_all else full_pred_df[full_pred_df['user_id'] == target_user_id]
                if data_subset.empty: continue

                is_hit = data_subset.apply(lambda row: row['true_label'] in row[pred_cols].values, axis=1)
                data_subset['is_hit'] = is_hit
                accuracy_per_location = data_subset.groupby('true_label')['is_hit'].mean().reset_index()
                accuracy_per_location['is_hit'] *= 100
                accuracy_per_location.columns = ['location_id', 'count']
                processed_gdfs[f'{prefix.upper()} {model}'] = locs_gdf.merge(accuracy_per_location, on='location_id', how='inner')
            except FileNotFoundError: print(f"Warning: File not found: {filepath}, skipping {model} model.")
            
    return processed_gdfs

def generate_combined_plot(processed_gdfs, main_title, output_file, fixed_extent=None):
    """
    Creates a 2x4 combined plot with independent scales and subplot labels.
    """
    fig, axes = plt.subplots(2, 4, figsize=(24, 12), constrained_layout=True)
    
    prefixes = ['DTEPR', 'EPR']
    models = ['LSTM', 'Mamba', 'MHSA']
    row_labels = ['a', 'b'] # Labels for each row

    cmap = plt.get_cmap('YlOrRd')
    cmap_acc = plt.get_cmap('YlGn')
    for i, prefix in enumerate(prefixes):
        axes_row = axes[i]
        
        # --- 1. Handle the 'True' plot and its scale ---
        ax_true = axes_row[0]
        true_title = f'{prefix} True'
        ax_true.set_facecolor('#ffffff')
        if 'switzerland_gdf' in globals(): switzerland_gdf.plot(ax=ax_true, color='white', edgecolor='black', zorder=1)
        if true_title in processed_gdfs:
            true_gdf = processed_gdfs[true_title]
            vmin = max(1e-4, np.percentile(true_gdf['count'], 5)); vmax = max(vmin + 0.1, np.percentile(true_gdf['count'], 95))
            norm = Normalize(vmin=vmin, vmax=vmax)
            true_gdf.plot(column='count', ax=ax_true, cmap=cmap, norm=norm, markersize=15, alpha=0.8, zorder=2)
            sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax_true, orientation='vertical', shrink=0.7, pad=0.04)
            cbar.set_label('Visit Frequency (%)', fontsize=10)
        ax_true.set_title(true_title, fontsize=14); ax_true.set_axis_off()
        if fixed_extent: ax_true.set_xlim(fixed_extent[0], fixed_extent[1]); ax_true.set_ylim(fixed_extent[2], fixed_extent[3])
        
        # --- CORE CHANGE: Add subplot label ---
        ax_true.text(0.05, 0.05, f'({row_labels[i]}1)', transform=ax_true.transAxes, fontsize=14, fontweight='bold')
        
        # --- 2. Handle the Prediction plots and their shared scale ---
        pred_axes = axes_row[1:]
        pred_gdfs = [processed_gdfs[f'{prefix} {m}'] for m in models if f'{prefix} {m}' in processed_gdfs]
        if pred_gdfs:
            norm_pred = Normalize(vmin=0, vmax=100)
            for j, model in enumerate(models):
                ax_pred = pred_axes[j]
                pred_title = f'{prefix} {model}'
                ax_pred.set_facecolor('#ffffff')
                if 'switzerland_gdf' in globals(): switzerland_gdf.plot(ax=ax_pred, color='white', edgecolor='black', zorder=1)
                if pred_title in processed_gdfs:
                    processed_gdfs[pred_title].plot(column='count', ax=ax_pred, cmap=cmap_acc, norm=norm_pred, markersize=15, alpha=0.8, zorder=2)
                ax_pred.set_title(pred_title, fontsize=14); ax_pred.set_axis_off()
                if fixed_extent: ax_pred.set_xlim(fixed_extent[0], fixed_extent[1]); ax_pred.set_ylim(fixed_extent[2], fixed_extent[3])
                
                # --- CORE CHANGE: Add subplot label ---
                ax_pred.text(0.05, 0.05, f'({row_labels[i]}{j+2})', transform=ax_pred.transAxes, fontsize=14, fontweight='bold')

            sm_pred = ScalarMappable(cmap=cmap_acc, norm=norm_pred); sm_pred.set_array([])
            cbar_pred = fig.colorbar(sm_pred, ax=pred_axes, orientation='vertical', shrink=0.7, pad=0.04)
            cbar_pred.set_label(f'Top-{TOP_K} Hit Rate (%)', fontsize=10)
            
    fig.suptitle(main_title, fontsize=24)
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nCombined image saved to: {output_file}"); plt.show()

In [ ]:
# 1. Load geographic data
locs_gdf = prepare_locations_data(LOCATIONS_FILE)
switzerland_gdf = get_switzerland_boundaries(SWITZERLAND_GEOJSON)

# 2. Load and process data for the two different metrics
processed_gdfs = load_and_process_data(
    pred_dir=PREDICTIONS_DIR, 
    locs_gdf=locs_gdf, 
    analyze_all=ANALYZE_ALL_USERS, 
    target_user_id=TARGET_USER_ID,
    top_k=TOP_K
)

# 3. Fix the map extent to the full view of Switzerland for consistency
if switzerland_gdf is not None:
    bounds = switzerland_gdf.total_bounds
    x_range = bounds[2] - bounds[0]; y_range = bounds[3] - bounds[1]
    padding = 0.05
    GLOBAL_EXTENT = (
        bounds[0] - x_range * padding, bounds[2] + x_range * padding,
        bounds[1] - y_range * padding, bounds[3] + y_range * padding
    )
    print(f"Map extent has been fixed to the full view of Switzerland.")
else:
    GLOBAL_EXTENT = None

In [ ]:
# Dynamically generate title and filename based on the analysis mode
if ANALYZE_ALL_USERS:
    main_title_suffix = f'Acc@{TOP_K}'
    output_filename_suffix = f'{TOP_K}'
else:
    main_title_suffix = f'(User ID: {TARGET_USER_ID}, Acc@{TOP_K})'
    output_filename_suffix = f'user_{TARGET_USER_ID}_ACC@{TOP_K}'

# Generate the visualization
if processed_gdfs:
    generate_combined_plot(
        processed_gdfs=processed_gdfs,
        main_title=f'Prediction {main_title_suffix} on each location in test set',
        output_file=f'{OUTPUT_FILE_PREFIX}_{output_filename_suffix}.png',
        fixed_extent=GLOBAL_EXTENT
    )
else:
    print("No data available to visualize.")